# Lab 03 — Dự báo phụ tải điện theo chuỗi thời gian

Notebook thực hành dự báo một giờ tiếp theo cho chuỗi phụ tải AEP (`AEP_MW`). Notebook dùng dữ liệu thật `data/raw/AEP_hourly.csv` và các bộ ước lượng NumPy tự cài đặt trong `src/lab03`; không tự chuyển sang dữ liệu giả nếu thiếu file.

**Quy trình:** nạp/làm sạch dữ liệu → tạo đặc trưng chỉ từ quá khứ → chia Train/Validation/Test theo thời gian → walk-forward CV trên Train → chọn mô hình bằng Validation → đánh giá một lần trên Test → dự báo đệ quy 24 giờ.

> Chạy lần lượt từ trên xuống bằng **Run All**. Không chạy ngược các cell phụ thuộc biến ở cell trước.

## 0. Chuẩn bị môi trường

Chạy tại thư mục gốc repository. Nếu chưa cài môi trường, mở terminal tại thư mục gốc và chạy `python -m pip install -r requirements.txt`, sau đó `jupyter lab` hoặc `jupyter notebook`.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "lab03").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy src/lab03. Hãy mở notebook trong repository Lab_TimeSeries.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lab03.data import load_hourly_series, chronological_split
from src.lab03.features import build_supervised_features, recursive_forecast
from src.lab03.cv import evaluate_walk_forward
from src.lab03.metrics import regression_metrics
from src.lab03.lab03_utils.report import write_report_bundle
from src.lab03.lab03_models import (
    NaiveLastValue, LinearRegressionScratch, DecisionTreeRegressorScratch,
    RandomForestRegressorScratch, XGBoostScratch,
)

DATA_PATH = ROOT / "data" / "raw" / "AEP_hourly.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Thiếu dữ liệu AEP: {DATA_PATH}. Đặt AEP_hourly.csv vào data/raw/ rồi chạy lại.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ROOT / "reports" / "lab03" / "notebook_runs" / RUN_ID
FIGURE_DIR = RUN_DIR / "figures"
MODEL_DIR = RUN_DIR / "models"
for directory in (RUN_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)
print("Run ID:", RUN_ID)
print("Dataset:", DATA_PATH)

## 1. Nạp và kiểm tra dữ liệu

Dữ liệu được sắp xếp theo thời gian, loại timestamp trùng, đưa về lưới một giờ và chỉ nội suy các khoảng trống nội bộ tối đa ba giờ. Khoảng trống dài hơn sẽ báo lỗi để tránh tự tạo dữ liệu không có căn cứ.

In [ ]:
series, ingestion = load_hourly_series(DATA_PATH)
display(pd.Series(ingestion, name="value").to_frame())
display(series.describe().to_frame("AEP_MW").round(2))
assert isinstance(series.index, pd.DatetimeIndex)
assert series.index.is_monotonic_increasing and series.index.is_unique
assert series.notna().all()
print(f"Dữ liệu sạch: {len(series):,} giờ | {series.index.min()} → {series.index.max()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
series.plot(ax=axes[0], linewidth=0.45, title="AEP hourly load — toàn bộ chuỗi")
axes[0].set_ylabel("AEP_MW")
series.loc["2016-01-04":"2016-01-17"].plot(ax=axes[1], linewidth=1.0, title="Hai tuần minh họa")
axes[1].set_ylabel("AEP_MW")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_hourly_series.png", dpi=140, bbox_inches="tight")
plt.show()

## 2. Tạo đặc trưng dự báo một bước

Đặc trưng gồm các lag 1–168 giờ, trung bình/độ lệch chuẩn trượt 6–168 giờ và lịch tuần hoàn. Hàm tạo rolling dùng `shift(1)`, do đó không lấy giá trị mục tiêu tại thời điểm dự báo.

In [ ]:
LAGS = (1, 2, 3, 6, 12, 24, 48, 168)
ROLLING_WINDOWS = (6, 12, 24, 168)
X, y = build_supervised_features(
    series, lags=LAGS, rolling_windows=ROLLING_WINDOWS, include_calendar=True
)
display(X.head(3).round(2))
print("X:", X.shape, "y:", y.shape, "| số đặc trưng:", X.shape[1])

timestamp = X.index[200]
position = series.index.get_loc(timestamp)
assert X.loc[timestamp, "lag_1"] == series.iloc[position - 1]
assert np.isclose(X.loc[timestamp, "roll_mean_6"], series.iloc[position - 6:position].mean())
assert y.loc[timestamp] == series.loc[timestamp]
print("Kiểm tra causal lag/rolling: đạt.")

## 3. Chia Train / Validation / Test theo thời gian

Tỷ lệ 70/15/15, không shuffle. Validation dùng chọn mô hình; Test được giữ riêng đến bước đánh giá cuối.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = chronological_split(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15
)
split_summary = pd.DataFrame([
    {"split": name, "rows": len(target), "start": target.index.min(),
     "end": target.index.max(), "target_mean": target.mean()}
    for name, target in (("Train", y_train), ("Validation", y_val), ("Test", y_test))
]).set_index("split")
display(split_summary)
assert X_train.index.max() < X_val.index.min() < X_test.index.min()

fig, ax = plt.subplots(figsize=(13, 3.5))
for name, target in (("Train", y_train), ("Validation", y_val), ("Test", y_test)):
    ax.plot(target.index, target, linewidth=0.5, label=name)
ax.set_title("Chia tập theo thứ tự thời gian")
ax.set_ylabel("AEP_MW")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_temporal_split.png", dpi=140, bbox_inches="tight")
plt.show()

## 4. Walk-forward cross-validation trên Train

Năm fold mở rộng theo thời gian, chỉ dùng Train. Các cấu hình cây được giới hạn vừa phải để notebook chạy được trên máy cá nhân; đây là cấu hình thực hành, không phải tuning sản xuất.

In [ ]:
MODEL_FACTORIES = {
    "Naive Last-Value": lambda: NaiveLastValue(),
    "Linear Regression": lambda: LinearRegressionScratch(ridge=1e-6),
    "CART": lambda: DecisionTreeRegressorScratch(
        max_depth=5, min_samples_split=30, min_samples_leaf=15,
        max_thresholds=8, random_state=42, max_train_rows=2500),
    "Random Forest": lambda: RandomForestRegressorScratch(
        n_estimators=5, max_depth=5, min_samples_split=30, min_samples_leaf=15,
        max_features="sqrt", max_thresholds=8, random_state=42, max_train_rows=2500),
    "XGBoost-style": lambda: XGBoostScratch(
        n_estimators=12, learning_rate=0.06, max_depth=3, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, max_bins=12,
        random_state=42, max_train_rows=5000),
}

cv_detail, cv_aggregate = evaluate_walk_forward(
    X_train, y_train, MODEL_FACTORIES, n_splits=5
)
display(cv_detail.groupby("model")["RMSE"].agg(["mean", "std"]).sort_values("mean").round(2))
display(cv_detail.pivot(index="fold", columns="model", values="RMSE").round(2))

## 5. Chọn mô hình bằng Validation

Mỗi ứng viên được fit trên Train và chấm trên cùng một Validation. Chốt champion bằng Validation RMSE trước khi xem Test.

In [ ]:
validation_rows = []
for name, factory in MODEL_FACTORIES.items():
    model = factory()
    fit_start = time.perf_counter()
    model.fit(X_train.to_numpy(dtype=float), y_train.to_numpy(dtype=float))
    fit_seconds = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    prediction = model.predict(X_val.to_numpy(dtype=float))
    predict_seconds = time.perf_counter() - predict_start
    validation_rows.append({"model": name, **regression_metrics(y_val, prediction),
                            "fit_seconds": fit_seconds, "predict_seconds": predict_seconds})
validation_summary = pd.DataFrame(validation_rows).set_index("model").sort_values("RMSE")
display(validation_summary.round(3))
CHAMPION_NAME = str(validation_summary.index[0])
print("Champion theo Validation RMSE:", CHAMPION_NAME)

## 6. Đánh giá trên Test chưa từng dùng để chọn mô hình

Fit lại mỗi ứng viên trên Train + Validation rồi đánh giá trên Test. Bảng Test chỉ dùng báo cáo hiệu năng; không thay đổi champion đã chốt.

In [ ]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])
test_predictions = {}
final_models = {}
final_fit_seconds = {}
test_predict_seconds = {}
test_rows = []
for name, factory in MODEL_FACTORIES.items():
    model = factory()
    fit_start = time.perf_counter()
    model.fit(X_train_val.to_numpy(dtype=float), y_train_val.to_numpy(dtype=float))
    final_fit_seconds[name] = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    prediction = np.asarray(model.predict(X_test.to_numpy(dtype=float)), dtype=float)
    test_predict_seconds[name] = time.perf_counter() - predict_start
    final_models[name] = model
    test_predictions[name] = prediction
    test_rows.append({"model": name, **regression_metrics(y_test, prediction)})
test_summary = pd.DataFrame(test_rows).set_index("model")
test_summary.insert(0, "validation_RMSE", validation_summary["RMSE"])
display(test_summary.sort_values("validation_RMSE").round(3))
print("Champion vẫn được xác định từ Validation:", CHAMPION_NAME)

In [ ]:
n_show = min(24 * 7, len(y_test))
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(y_test.index[:n_show], y_test.iloc[:n_show], label="Actual", linewidth=1.5)
axes[0].plot(y_test.index[:n_show], test_predictions[CHAMPION_NAME][:n_show],
             label=f"{CHAMPION_NAME} prediction", linewidth=1.1)
axes[0].set_title("Dự báo một bước — tuần đầu của Test")
axes[0].legend()
residual = y_test.to_numpy(dtype=float) - test_predictions[CHAMPION_NAME]
axes[1].plot(y_test.index, residual, linewidth=0.45)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Residual trên toàn Test — {CHAMPION_NAME}")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_test_and_residuals.png", dpi=140, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
test_summary.sort_values("RMSE")["RMSE"].plot.bar(ax=ax)
ax.set_title("So sánh RMSE trên tập Test (thấp hơn tốt hơn)")
ax.set_ylabel("RMSE (MW)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_test_model_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

## 7. Dự báo đệ quy 24 giờ tiếp theo

Đây là dự báo tại một mốc cố định: mô hình champion được fit lại trên toàn bộ dữ liệu có nhãn, sau đó các dự báo được đưa ngược vào lịch sử cho bước kế tiếp. Không dùng giá trị thực tương lai.

In [ ]:
forecast_model = MODEL_FACTORIES[CHAMPION_NAME]()
forecast_model.fit(X.to_numpy(dtype=float), y.to_numpy(dtype=float))
forecast = recursive_forecast(
    forecast_model, series, horizon=24, feature_names=X.columns,
    lags=LAGS, rolling_windows=ROLLING_WINDOWS, include_calendar=True
)
display(forecast.to_frame("forecast_AEP_MW").round(2))
assert len(forecast) == 24 and forecast.index.min() > series.index.max()

fig, ax = plt.subplots(figsize=(12, 4.5))
series.iloc[-24 * 7:].plot(ax=ax, label="Observed", linewidth=1.3)
forecast.plot(ax=ax, label="24-hour recursive forecast", marker="o", linewidth=1.2)
ax.set_title(f"Dự báo 24 giờ — champion: {CHAMPION_NAME}")
ax.set_ylabel("AEP_MW")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_future_24h_forecast.png", dpi=140, bbox_inches="tight")
plt.show()

## 8. Tạo report và model summary

Mỗi lần chạy lưu đầy đủ kết quả trong `reports/lab03/notebook_runs/<run_id>/` và cập nhật các bản mới nhất tại `reports/lab03/` cùng `reports/model_summary/lab03/`. Bộ report gồm Markdown, HTML, PDF; model summary gồm CSV, JSON, Markdown và các bảng metric.

In [ ]:
REPORT_ROOT = ROOT / "reports" / "lab03"
SUMMARY_ROOT = ROOT / "reports" / "model_summary" / "lab03"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

model_files = {}
for name, model in final_models.items():
    slug = name.lower().replace(" ", "_").replace("-", "_")
    model_path = MODEL_DIR / f"{slug}.pkl"
    with model_path.open("wb") as file:
        pickle.dump(model, file)
    model_files[name] = model_path
with (MODEL_DIR / "champion_future_forecast.pkl").open("wb") as file:
    pickle.dump(forecast_model, file)

series.rename("AEP_MW").to_csv(RUN_DIR / "cleaned_hourly_series.csv", header=True)
cv_detail.to_csv(RUN_DIR / "cv_fold_metrics.csv", index=False)
validation_summary.to_csv(RUN_DIR / "validation_metrics.csv")
test_summary.to_csv(RUN_DIR / "test_metrics.csv")
pd.DataFrame({"actual": y_test, **test_predictions}).to_csv(RUN_DIR / "test_predictions.csv")
forecast.to_csv(RUN_DIR / "future_24h_forecast.csv", header=True)

metric_names = ["MAE", "MSE", "RMSE", "MAPE (%)", "sMAPE (%)", "R2"]
cv_stats = cv_detail.groupby("model")[metric_names].agg(["mean", "std"])
summary_rows = []
for name in MODEL_FACTORIES:
    row = {
        "run_id": RUN_ID,
        "model": name,
        "selected_champion": name == CHAMPION_NAME,
        "feature_count": int(X.shape[1]),
        "total_train_validation_rows": int(len(X_train_val)),
        "training_rows_used": int(getattr(final_models[name], "n_training_rows_", len(X_train_val))),
        "fit_seconds": float(final_fit_seconds[name]),
        "predict_seconds_test": float(test_predict_seconds[name]),
        "model_file": model_files[name].relative_to(ROOT).as_posix(),
        "model_parameters": json.dumps(final_models[name].get_params(), sort_keys=True),
    }
    for metric in metric_names:
        row[f"cv_{metric}_mean"] = float(cv_stats.loc[name, (metric, "mean")])
        row[f"cv_{metric}_std"] = float(cv_stats.loc[name, (metric, "std")])
        row[f"validation_{metric}"] = float(validation_summary.loc[name, metric])
        row[f"test_{metric}"] = float(test_summary.loc[name, metric])
    summary_rows.append(row)
model_summary = pd.DataFrame(summary_rows).sort_values("validation_RMSE")

summary_columns = [
    "run_id", "model", "selected_champion", "feature_count",
    "total_train_validation_rows", "training_rows_used", "fit_seconds",
    "predict_seconds_test", "model_file", "model_parameters",
    *[f"cv_{metric}_{stat}" for metric in metric_names for stat in ("mean", "std")],
    *[f"validation_{metric}" for metric in metric_names],
    *[f"test_{metric}" for metric in metric_names],
]
model_summary = model_summary[summary_columns]
model_summary_json = {
    "run_id": RUN_ID,
    "champion_selected_by": "minimum validation_RMSE",
    "champion": CHAMPION_NAME,
    "models": summary_rows,
}
validation_json = {
    "run_id": RUN_ID,
    "selection_metric": "RMSE",
    "models": validation_summary.reset_index().to_dict(orient="records"),
}
test_json = {
    "run_id": RUN_ID,
    "champion_selected_by_validation": CHAMPION_NAME,
    "models": test_summary.reset_index().to_dict(orient="records"),
}

def write_json(path, payload):
    with Path(path).open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2,
                  default=lambda value: value.item() if isinstance(value, np.generic) else str(value),
                  allow_nan=False)

def write_model_summary(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model_summary.to_csv(directory / "model_summary.csv", index=False)
    write_json(directory / "model_summary.json", model_summary_json)
    summary_view = model_summary[[
        "model", "selected_champion", "cv_RMSE_mean", "cv_RMSE_std",
        "validation_RMSE", "test_MAE", "test_RMSE", "test_MAPE (%)",
        "test_R2", "fit_seconds", "training_rows_used",
    ]]
    summary_lines = [
        "# Lab 03 Model Summary",
        "",
        f"Run: `{RUN_ID}`",
        "",
        f"Champion selected by Validation RMSE: **{CHAMPION_NAME}**.",
        "",
        "Test metrics are for held-out Test and did not participate in selection.",
        "",
        "| " + " | ".join(str(column) for column in summary_view.columns) + " |",
        "| " + " | ".join("---" for _ in summary_view.columns) + " |",
    ]
    for values in summary_view.itertuples(index=False, name=None):
        cells = []
        for value in values:
            if pd.isna(value):
                cells.append("N/A")
            elif isinstance(value, (float, np.floating)):
                cells.append(f"{value:.3f}")
            else:
                cells.append(str(value).replace("|", "\\|"))
        summary_lines.append("| " + " | ".join(cells) + " |")
    summary_lines += ["", "See `MODEL_SUMMARY_SPEC.md` in `docs/lab03/` for field definitions."]
    model_summary_md = "\n".join(summary_lines) + "\n"
    (directory / "model_summary.md").write_text(model_summary_md, encoding="utf-8")
    cv_detail.to_csv(directory / "cv_fold_metrics.csv", index=False)
    write_json(directory / "validation_metrics.json", validation_json)
    write_json(directory / "test_metrics.json", test_json)

write_model_summary(RUN_DIR)
write_model_summary(SUMMARY_ROOT)

cv_summary = []
for name in MODEL_FACTORIES:
    cv_summary.append({
        "model": name,
        "RMSE_mean": float(cv_stats.loc[name, ("RMSE", "mean")]),
        "RMSE_std": float(cv_stats.loc[name, ("RMSE", "std")]),
        "MAE_mean": float(cv_stats.loc[name, ("MAE", "mean")]),
        "R2_mean": float(cv_stats.loc[name, ("R2", "mean")]),
    })
report_model_rows = []
for row in summary_rows:
    report_model_rows.append({
        "model": row["model"],
        "cv_RMSE_mean": row["cv_RMSE_mean"],
        "validation_RMSE": row["validation_RMSE"],
        "test_MAE": row["test_MAE"],
        "test_RMSE": row["test_RMSE"],
        "test_MAPE (%)": row["test_MAPE (%)"],
        "test_R2": row["test_R2"],
        "fit_seconds": row["fit_seconds"],
    })
figure_paths = {
    "hourly_series": FIGURE_DIR / "01_hourly_series.png",
    "temporal_split": FIGURE_DIR / "02_temporal_split.png",
    "test_and_residuals": FIGURE_DIR / "03_test_and_residuals.png",
    "test_model_comparison": FIGURE_DIR / "04_test_model_comparison.png",
    "future_24h_forecast": FIGURE_DIR / "05_future_24h_forecast.png",
}
relative_figures = {name: path.relative_to(REPORT_ROOT).as_posix()
                    for name, path in figure_paths.items()}
report_payload = {
    "run_id": RUN_ID,
    "status": "SUCCESS",
    "champion": CHAMPION_NAME,
    "summary_text": (
        f"On {ingestion['dataset_name']}, {len(series):,} cleaned hourly observations were used. "
        f"{CHAMPION_NAME} was selected using Validation RMSE "
        f"({validation_summary.loc[CHAMPION_NAME, 'RMSE']:.2f} MW). Its held-out Test RMSE was "
        f"{test_summary.loc[CHAMPION_NAME, 'RMSE']:.2f} MW. A recursive {len(forecast)}-hour forecast was generated."
    ),
    "dataset": {**ingestion, "missing_values_remaining": int(series.isna().sum())},
    "split": {
        "train": {"rows": len(y_train), "start": str(y_train.index.min()), "end": str(y_train.index.max()), "ratio": 0.70},
        "validation": {"rows": len(y_val), "start": str(y_val.index.min()), "end": str(y_val.index.max()), "ratio": 0.15},
        "test": {"rows": len(y_test), "start": str(y_test.index.min()), "end": str(y_test.index.max()), "ratio": 0.15},
    },
    "feature_config": {"lags": list(LAGS), "rolling_windows": list(ROLLING_WINDOWS),
                       "include_calendar_features": True},
    "feature_count": int(X.shape[1]),
    "cv_summary": cv_summary,
    "n_splits": 5,
    "model_rows": report_model_rows,
    "forecast_horizon": len(forecast),
    "figures": {name: str(path) for name, path in figure_paths.items()},
    "figure_relative_paths": {name: f"figures/{path.name}" for name, path in figure_paths.items()},
    "legacy_snapshot": [],
}
run_report_paths = write_report_bundle(report_payload, RUN_DIR)
latest_report_payload = dict(report_payload)
latest_report_payload["figure_relative_paths"] = relative_figures
latest_report_paths = write_report_bundle(latest_report_payload, REPORT_ROOT)

run_summary = {
    "run_id": RUN_ID,
    "dataset": ingestion,
    "feature_count": int(X.shape[1]),
    "split_rows": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "champion_selected_by_validation": CHAMPION_NAME,
    "validation_metrics": validation_summary.loc[CHAMPION_NAME].to_dict(),
    "test_metrics": test_summary.loc[CHAMPION_NAME].drop(labels="validation_RMSE").to_dict(),
    "forecast_horizon_hours": len(forecast),
    "forecast_start": str(forecast.index.min()),
    "forecast_end": str(forecast.index.max()),
    "figure_dir": str(FIGURE_DIR),
    "reports": latest_report_paths,
    "model_summary_dir": str(SUMMARY_ROOT),
}
write_json(RUN_DIR / "run_summary.json", run_summary)
write_json(ROOT / "data" / "processed" / "lab03" / "run_result.json", run_summary)

print("Hoàn tất Lab 03.")
print("Champion:", CHAMPION_NAME)
print("Report files:", latest_report_paths)
print("Model summary:", SUMMARY_ROOT)
print("Per-run outputs:", RUN_DIR)
print("Test metrics:", test_summary.loc[CHAMPION_NAME].to_dict())
print("Forecast:", forecast.index.min(), "→", forecast.index.max())

## Câu hỏi thảo luận

1. Vì sao không được dùng Test để chọn champion hoặc tinh chỉnh siêu tham số?
2. So sánh Naive, Linear Regression và các mô hình cây theo Validation/Test. Mô hình nào tổng quát hóa tốt hơn?
3. Vì sao kết quả dự báo một bước trên Test khác với dự báo đệ quy 24 giờ tại một mốc cố định?
4. Lag/rolling nào có thể nắm bắt tính mùa vụ ngày và tuần? Nêu một cách kiểm tra thêm mà không làm rò rỉ dữ liệu tương lai.

Notebook này dùng API Lab03 hiện hành; đây là notebook chạy chính cho dự án.